# 02 · Crudo → Plata: limpieza, calidad y cuarentena

El paso `ranking` deja una fila por empresa, validada con pandera. Nada se
borra en silencio: cada decisión queda como marca, en la cuarentena o en el
reporte de calidad.

> **Una sola fuente de verdad.** Las funciones de este notebook viven en
> `src/etl/`, se prueban con pytest y son las mismas que corre `etl` en la
> terminal y en CI. Aquí se las llama y se mira el resultado: así el notebook
> no se desactualiza cuando se corrige el código.

In [1]:
# El paquete `etl` se instala con `uv sync` en etl/.venv. Si el kernel del
# notebook no es ese entorno, se agrega src/ a la ruta como respaldo.
import sys
from pathlib import Path

ETL = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:
    import etl  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(ETL / "src"))

import json

import pandas as pd

from etl import config
from etl.comun import configurar_registro

configurar_registro()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

ANIO = config.ANIO_POR_DEFECTO
ANIO

2025

## 1. Correr el paso

Es idempotente: volver a correrlo reescribe el Parquet del año y da el mismo
resultado.

In [2]:
from etl.pasos import ranking

plata = ranking.ejecutar(ANIO)
print(f"{len(plata):,} empresas en plata")
plata.head(3)

06:03:11  INFO    Ranking 2025: 154,813 filas crudas, corte Thu Oct 22 00:01:13 ECT 2026


06:03:15  INFO    Plata 2025: 154,687 empresas -> /home/miguel/Documents/EPN/Tesis/observatorio-epn/etl/datos/plata/2025/ranking.parquet


154,687 empresas en plata


,posicion,anio,posicion_anterior,anio_anterior,expediente,nombre,tipo_compania,region,provincia,ciudad,...,ingreso_total_no_reportado,utilidad_no_reportada,motivo_descarte,tamano_derivado,ventas_por_empleado,veces_mediana_division,alerta_empleados,ruc_original,ruc_valido,ruc
0,0,0,140765,2024,334900,M&B GROUPWINNERS S.A.S.,SOCIEDAD POR ACCIONES SIMPLIFICADA,SIERRA,PICHINCHA,QUITO,...,False,False,,micro_pequena,0.0000,0.000000,,1793196194001,True,1793196194001
1,0,0,81152,2024,353583,INCOMARSA S.A.S.,SOCIEDAD POR ACCIONES SIMPLIFICADA,COSTA,GUAYAS,GUAYAQUIL,...,False,False,,micro_pequena,8911.3625,0.104608,,0993385165001,True,0993385165001
2,0,0,0,0,305653,REFRIGERACION INDUSTRIAL CORNEJO REFRICOR-EC CIA.LTDA.,RESPONSABILIDAD LIMITADA,COSTA,GUAYAS,GUAYAQUIL,...,False,False,,mediana,253553.6900,3.798749,,0993004766001,True,0993004766001


## 2. El reporte de calidad del año

In [3]:
calidad = json.loads((config.ruta_plata(ANIO) / "calidad_ranking.json").read_text())
print(json.dumps(calidad, indent=2, ensure_ascii=False))

{
  "anio": 2025,
  "fuente": "crudo/2025/ranking_2025.xlsx",
  "sha256_fuente": "0f01e577c720e98e2b21804a725ace5758093e731c0ab1723e9abc9f803b4059",
  "fecha_corte": "Thu Oct 22 00:01:13 ECT 2026",
  "filas_crudo": 154813,
  "filas_identicas_quitadas": 116,
  "expedientes_con_conflicto": [
    55002,
    56726,
    102297,
    130500,
    135453,
    176088,
    300724,
    703139,
    709214,
    743120
  ],
  "filas_descartadas": 0,
  "motivos_de_descarte": {},
  "filas_plata": 154687,
  "cifras_en_cero_pct": {
    "empleados": 0.0,
    "activo": 3.05,
    "patrimonio": 2.66,
    "ventas": 38.08,
    "utilidad_neta": 34.59,
    "ingreso_total": 36.71
  },
  "no_reportado": {
    "ingreso_total": 0,
    "utilidad_neta": 5051
  },
  "marcas": {
    "anio_cero": 30,
    "nueva_en_ranking": 22548,
    "sin_ventas": 58905,
    "sin_actividad": 4353,
    "patrimonio_negativo": 17488,
    "revisar_utilidad": 203
  },
  "tamano_derivado": {
    "micro_pequena": 140309,
    "mediana": 10041,


## 3. Cuarentena: las filas que no entraron

Cifras negativas donde no pueden serlo, o ventas mayores que el ingreso total
habiéndose reportado el ingreso. Se guardan aparte con el motivo para poder
rendir cuentas de cada fila del archivo original.

In [4]:
ruta = config.ruta_plata(ANIO) / "descartes_ranking.csv"
if ruta.exists():
    descartes = pd.read_csv(ruta)
    display(descartes["motivo_descarte"].value_counts().to_frame("filas"))
    display(descartes[["expediente", "nombre", "ventas", "ingreso_total", "motivo_descarte"]].head())
else:
    print(f"El año {ANIO} no dejó filas en cuarentena.")

El año 2025 no dejó filas en cuarentena.


## 4. Marcas de calidad

La marca no corrige el dato: lo señala. `anio_cero` es la que excluye filas
del universo; las demás viajan hasta la ficha para que la API pueda advertir.

In [5]:
marcas = ["anio_cero", "nueva_en_ranking", "sin_ventas", "sin_actividad",
          "patrimonio_negativo", "revisar_utilidad",
          "ingreso_total_no_reportado", "utilidad_no_reportada"]
pd.DataFrame(
    {"filas": [int(plata[m].sum()) for m in marcas],
     "%": [round(float(plata[m].mean() * 100), 2) for m in marcas]},
    index=marcas,
)

,filas,%
anio_cero,30,0.02
nueva_en_ranking,22548,14.58
sin_ventas,58905,38.08
sin_actividad,4353,2.81
patrimonio_negativo,17488,11.31
revisar_utilidad,203,0.13
ingreso_total_no_reportado,0,0.00
utilidad_no_reportada,5051,3.27


## 5. Tamaño derivado de las ventas

La columna oficial de tamaño viene vacía en el ranking, así que se deriva de
las ventas anuales con los umbrales del COPCI.

In [6]:
print(f"mediana: > USD {config.UMBRAL_MEDIANA:,}   ·   grande: > USD {config.UMBRAL_GRANDE:,}")
plata["tamano_derivado"].value_counts().to_frame("empresas")

mediana: > USD 1,000,000   ·   grande: > USD 5,000,000


,empresas
tamano_derivado,
micro_pequena,140309
mediana,10041
grande,4337


## 6. La alerta de empleados, con su causa

In [7]:
display(plata["alerta_empleados"].replace("", "sin alerta").value_counts().to_frame("empresas"))
plata[plata["alerta_empleados"] != ""].nlargest(8, "ventas")[
    ["nombre", "empleados", "ventas", "ventas_por_empleado", "alerta_empleados"]
]

,empresas
alerta_empleados,
sin alerta,152668
fuera_de_rango_division,1148
grande_menos_10,869
imposible,2


,nombre,empleados,ventas,ventas_por_empleado,alerta_empleados
33,AURELIAN ECUADOR S.A,1997,1.741957e+09,8.722868e+05,fuera_de_rango_division
31,ECUACORRIENTE S.A.,1491,1.664823e+09,1.116581e+06,fuera_de_rango_division
38,CORPORACION EL ROSADO S.A.,6,1.641395e+09,2.735658e+08,grande_menos_10
152,PRIMAX COMERCIAL DEL ECUADOR SOCIEDAD ANONIMA,136,1.397067e+09,1.027255e+07,fuera_de_rango_division
79,TIENDAS TUTI TTDE S.A.,8,1.008144e+09,1.260180e+08,grande_menos_10
744,PETROLEOS Y SERVICIOS PYS C.A.,62,1.007444e+09,1.624910e+07,fuera_de_rango_division
37,SHAYA ECUADOR S.A.,145,7.311721e+08,5.042566e+06,fuera_de_rango_division
122,VITAPRO ECUADOR CIA. LTDA.,248,6.822298e+08,2.750927e+06,fuera_de_rango_division


## 7. RUC desde el directorio de compañías

El ranking no trae RUC. Se cruza con `bi_compania.csv`, y como el directorio
trae RUC con letras o puntos, solo se acepta el de 13 dígitos.

In [8]:
print("con RUC válido:", int(plata["ruc_valido"].sum()))
print("sin RUC en el directorio:", int(plata["ruc_original"].isna().sum()))
print("formato inválido:", int((plata["ruc_original"].notna() & ~plata["ruc_valido"]).sum()))
plata[plata["ruc_original"].notna() & ~plata["ruc_valido"]][["expediente", "nombre", "ruc_original"]].head()

con RUC válido: 154684
sin RUC en el directorio: 0
formato inválido: 3


,expediente,nombre,ruc_original
48016,750870,SIDCAP ASESORÍA INTEGRAL Y CAPACITACIÓN PROFESIONAL S.A.S.,O195123391001
69913,754309,SALUD SOLIDARIA MÉDICAL CENTER S.A.,0993388798..1
127985,744918,TRANSOCT S.A.,099337330001


## 8. La validación es una condición, no un informe

Si el esquema no se cumple, el paso se detiene y no escribe. Esa es la
diferencia entre una capa plata confiable y una copia del crudo.

In [9]:
print(ranking.ESQUEMA)

<Schema DataFrameSchema(
    columns={
        'expediente': <Schema Column(name=expediente, type=DataType(int64))>
        'anio': <Schema Column(name=anio, type=DataType(int64))>
        'nombre': <Schema Column(name=nombre, type=DataType(string[pyarrow]))>
        'ciiu': <Schema Column(name=ciiu, type=DataType(string[pyarrow]))>
        'seccion_ciiu': <Schema Column(name=seccion_ciiu, type=DataType(string[pyarrow]))>
        'region': <Schema Column(name=region, type=DataType(string[pyarrow]))>
        'provincia': <Schema Column(name=provincia, type=DataType(string[pyarrow]))>
        'empleados': <Schema Column(name=empleados, type=DataType(int64))>
        'activo': <Schema Column(name=activo, type=DataType(float64))>
        'ventas': <Schema Column(name=ventas, type=DataType(float64))>
        'ingreso_total': <Schema Column(name=ingreso_total, type=DataType(float64))>
        'tamano_derivado': <Schema Column(name=tamano_derivado, type=DataType(string[pyarrow]))>
        'ru